# MountainCarContinuous SAC

## 实验目标

本实验使用 `SAC` 训练 `MountainCarContinuous-v0` 的连续动作控制策略。目标是让小车学会通过连续推力蓄能，最终稳定爬上右侧山顶，同时避免无效的大幅动作。

## 为什么这里选择 SAC

`MountainCarContinuous-v0` 是连续动作控制环境，`SAC` 通过熵正则保留探索，通常比单纯的确定性策略更稳，适合这类既要长期蓄能又要控制动作幅度的任务。

In [ ]:
from pathlib import Path

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

In [ ]:
ENV_ID = "MountainCarContinuous-v0"
TOTAL_TIMESTEPS = 180000
LEARNING_RATE = 3e-4
BUFFER_SIZE = 200000
LEARNING_STARTS = 5000
BATCH_SIZE = 256
GAMMA = 0.99
TAU = 0.005
TRAIN_FREQ = 1
GRADIENT_STEPS = 1
ENT_COEF = "auto"
EVAL_EPISODES = 100
ROLLOUT_EPISODES = 3
ROLLOUT_FPS = 30
MAX_STEPS = 999
SUCCESS_THRESHOLD_RETURN = 90
SEED = 42
DEVICE = "cpu"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

## 参数选择说明

这里的参数选择明显向“更容易收敛”倾斜：较大的回放池、更长的预热阶段、自动熵系数和较充分的训练预算。

In [ ]:
class EpisodeStatsCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_returns = []
        self.episode_lengths = []
        self.success_flags = []

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                r = float(info["episode"]["r"])
                l = int(info["episode"]["l"])
                self.episode_returns.append(r)
                self.episode_lengths.append(l)
                self.success_flags.append(int(r >= SUCCESS_THRESHOLD_RETURN))
        return True

In [ ]:
train_env = Monitor(gym.make(ENV_ID))
callback = EpisodeStatsCallback()

model = SAC(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=LEARNING_RATE,
    buffer_size=BUFFER_SIZE,
    learning_starts=LEARNING_STARTS,
    batch_size=BATCH_SIZE,
    gamma=GAMMA,
    tau=TAU,
    train_freq=TRAIN_FREQ,
    gradient_steps=GRADIENT_STEPS,
    ent_coef=ENT_COEF,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=0,
    device=DEVICE,
    seed=SEED,
)

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback, progress_bar=False)
train_env.close()

In [ ]:
episode_return_series = pd.Series(callback.episode_returns, dtype=float)
episode_length_series = pd.Series(callback.episode_lengths, dtype=float)
success_series = pd.Series(callback.success_flags, dtype=float)
rolling_return = episode_return_series.rolling(50).mean() if len(episode_return_series) >= 50 else episode_return_series
rolling_length = episode_length_series.rolling(50).mean() if len(episode_length_series) >= 50 else episode_length_series
rolling_success = success_series.rolling(50).mean() if len(success_series) >= 50 else success_series

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(callback.episode_returns, alpha=0.25, color="#4c72b0")
axes[0, 0].plot(rolling_return, color="#1f77b4")
axes[0, 0].axhline(SUCCESS_THRESHOLD_RETURN, linestyle="--", color="#2f2f2f", linewidth=1)
axes[0, 0].set_title("Episode Return")
axes[0, 1].plot(rolling_length, color="#55a868")
axes[0, 1].set_title("Rolling Episode Length")
axes[1, 0].plot(rolling_success, color="#c44e52")
axes[1, 0].set_ylim(0, 1.02)
axes[1, 0].set_title("Rolling Success Rate")
axes[1, 1].plot(pd.Series(callback.episode_returns).rolling(10).std(), color="#8172b3")
axes[1, 1].set_title("Rolling Return Std")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sac_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
eval_returns = []
eval_final_positions = []
eval_max_positions = []
eval_successes = []
eval_mean_abs_actions = []
for episode in range(EVAL_EPISODES):
    eval_env = gym.make(ENV_ID)
    obs, info = eval_env.reset(seed=SEED + 10000 + episode)
    total_reward = 0.0
    best_position = float(obs[0])
    actions = []
    for step in range(MAX_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        best_position = max(best_position, float(obs[0]))
        actions.append(abs(float(action[0])))
        if terminated or truncated:
            break
    eval_returns.append(total_reward)
    eval_final_positions.append(float(obs[0]))
    eval_max_positions.append(best_position)
    eval_successes.append(int(total_reward >= SUCCESS_THRESHOLD_RETURN))
    eval_mean_abs_actions.append(float(np.mean(actions)))
    eval_env.close()

eval_results = pd.DataFrame({"evaluation_return": eval_returns, "success": eval_successes, "final_position": eval_final_positions, "max_position": eval_max_positions, "mean_abs_action": eval_mean_abs_actions})
eval_results.head()

In [ ]:
position_values = np.linspace(-1.2, 0.6, 160)
velocity_values = np.linspace(-0.07, 0.07, 160)
action_grid = np.zeros((len(velocity_values), len(position_values)))
for i, velocity in enumerate(velocity_values):
    for j, position in enumerate(position_values):
        obs = np.array([position, velocity], dtype=np.float32)
        action, _ = model.predict(obs, deterministic=True)
        action_grid[i, j] = float(action[0])
plt.figure(figsize=(12, 5))
plt.imshow(action_grid, extent=[position_values[0], position_values[-1], velocity_values[0], velocity_values[-1]], origin="lower", aspect="auto", cmap="coolwarm")
plt.colorbar(label="Force")
plt.title("Deterministic Action Map")
plt.xlabel("Position")
plt.ylabel("Velocity")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sac_policy_visualization.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rollout_summaries = []
for rollout_idx in range(ROLLOUT_EPISODES):
    rollout_env = gym.make(ENV_ID, render_mode="rgb_array")
    obs, info = rollout_env.reset(seed=SEED + 20000 + rollout_idx)
    frames = []
    total_reward = 0.0
    best_position = float(obs[0])
    actions = []
    for step in range(MAX_STEPS):
        frame = rollout_env.render()
        if frame is not None:
            frames.append(frame)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = rollout_env.step(action)
        total_reward += reward
        best_position = max(best_position, float(obs[0]))
        actions.append(abs(float(action[0])))
        if terminated or truncated:
            break
    final_frame = rollout_env.render()
    if final_frame is not None:
        frames.extend([final_frame] * 20)
    rollout_summaries.append({"rollout_index": rollout_idx + 1, "seed": SEED + 20000 + rollout_idx, "total_reward": float(total_reward), "final_position": float(obs[0]), "max_position": float(best_position), "mean_abs_action": float(np.mean(actions)), "success": int(total_reward >= SUCCESS_THRESHOLD_RETURN)})
    if frames:
        imageio.mimsave(RESULTS_DIR / f"sac_rollout_{rollout_idx + 1}.gif", frames, fps=ROLLOUT_FPS)
    rollout_env.close()

pd.DataFrame(rollout_summaries)

In [ ]:
metrics = pd.DataFrame({"metric": ["device", "total_timesteps", "recent_episode_return_last_50", "recent_episode_length_last_50", "recent_success_rate_last_50", "evaluation_average_return", "evaluation_return_std", "evaluation_success_rate", "evaluation_average_final_position", "evaluation_average_max_position", "evaluation_average_mean_abs_action"], "value": [DEVICE, TOTAL_TIMESTEPS, float(episode_return_series.tail(50).mean()) if len(episode_return_series) > 0 else 0.0, float(episode_length_series.tail(50).mean()) if len(episode_length_series) > 0 else 0.0, float(success_series.tail(50).mean()) if len(success_series) > 0 else 0.0, float(np.mean(eval_returns)), float(np.std(eval_returns)), float(np.mean(eval_successes)), float(np.mean(eval_final_positions)), float(np.mean(eval_max_positions)), float(np.mean(eval_mean_abs_actions))]})
metrics

In [ ]:
episode_summary = pd.DataFrame({"episode_return": callback.episode_returns, "episode_length": callback.episode_lengths, "success": callback.success_flags})
episode_summary.to_csv(RESULTS_DIR / "sac_episode_summary.csv", index=False)
eval_results.to_csv(RESULTS_DIR / "sac_eval_results.csv", index=False)
metrics.to_csv(RESULTS_DIR / "sac_metrics.csv", index=False)
pd.DataFrame(rollout_summaries).to_csv(RESULTS_DIR / "sac_rollout_summary.csv", index=False)
print(metrics.to_string(index=False))